In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Load Clean Data ───────────────────────────────────────
df = pd.read_csv('../data/processed/clean_data.csv')
print(f"Loaded: {df.shape[0]:,} rows | {df.shape[1]:,} columns")

original_cols = df.shape[1]

Loaded: 590,540 rows | 204 columns


In [2]:
# ============================================================
# FEATURE 1: TIME-BASED FEATURES
# Source: EDA showed fraud peaks at hour 7-8 AM Day of week was relatively flat but still useful
# ============================================================

# Already computed in EDA — verify they exist
if 'hour' not in df.columns:
    df['hour'] = (df['TransactionDT'] // 3600) % 24
if 'day' not in df.columns:
    df['day']  = (df['TransactionDT'] // (3600 * 24)) % 7

# Is it the fraud peak hour? (EDA showed 6-9 AM peak)
df['is_peak_fraud_hour'] = df['hour'].between(6, 9).astype(int)

# Time of day buckets
def time_bucket(hour):
    if 0 <= hour < 6:   return 0  # Late night
    elif 6 <= hour < 12: return 1  # Morning (fraud peak)
    elif 12 <= hour < 18: return 2  # Afternoon
    else:                return 3  # Evening

df['time_of_day'] = df['hour'].apply(time_bucket)

# Is weekend?
df['is_weekend'] = (df['day'] >= 5).astype(int)

print("\n[FEATURE 1] Time features created:")
print(f"  is_peak_fraud_hour | is_weekend | time_of_day")
print(df[['hour', 'is_peak_fraud_hour', 'time_of_day', 'is_weekend']].head(5).to_string())


[FEATURE 1] Time features created:
  is_peak_fraud_hour | is_weekend | time_of_day
   hour  is_peak_fraud_hour  time_of_day  is_weekend
0     0                   0            0           0
1     0                   0            0           0
2     0                   0            0           0
3     0                   0            0           0
4     0                   0            0           0


In [3]:
# ============================================================
# FEATURE 2: TRANSACTION AMOUNT FEATURES
# Source: EDA showed fraud transactions have higher median amount — amount buckets will capture this
# ============================================================

# Amount buckets based on distribution
df['amt_bucket'] = pd.cut(df['TransactionAmt'],bins=[0, 50, 100, 500, 1000, np.inf],labels=[0, 1, 2, 3, 4]).astype(int)

# Amount is round number — fraudsters often use exact amounts
df['is_round_amount'] = (df['TransactionAmt'] % 1 == 0).astype(int)

# High value transaction flag (above 75th percentile)
amt_75 = df['TransactionAmt'].quantile(0.75)
df['is_high_value'] = (df['TransactionAmt'] > amt_75).astype(int)

print(f"\n[FEATURE 2] Amount features created:")
print(f"  75th percentile threshold: ${amt_75:.2f}")
print(f"  Round amount transactions: {df['is_round_amount'].sum():,}")
print(f"  High value transactions  : {df['is_high_value'].sum():,}")

# Fraud rate by amount bucket — validate signal
if 'isFraud' in df.columns:
    print("\n  Fraud rate by amount bucket:")
    print(df.groupby('amt_bucket')['isFraud'].mean().mul(100).round(2).to_string())


[FEATURE 2] Amount features created:
  75th percentile threshold: $125.00
  Round amount transactions: 305,013
  High value transactions  : 145,828

  Fraud rate by amount bucket:
amt_bucket
0    3.83
1    2.92
2    3.53
3    5.31
4    2.46


In [4]:
# ============================================================
# FEATURE 3: PRODUCT CODE RISK SCORE
# Source: EDA showed ProductCD 'C' has 11.69% fraud rate vs overall 3.5% — 3x higher risk
# Strategy: Map each product to its fraud rate from EDA
# ============================================================

# These rates came directly from your Phase 2 EDA output
product_fraud_rate = {
    'C': 0.1169,
    'S': 0.0590,
    'H': 0.0477,
    'R': 0.0378,
    'W': 0.0204
}

# ProductCD was label encoded in Phase 3
# Re-map using original fraud rates via frequency in data
# Since it's already encoded, create risk score from
# grouped fraud rate directly
product_risk = df.groupby('ProductCD')['isFraud'].mean()
df['product_risk_score'] = df['ProductCD'].map(product_risk)

print(f"\n[FEATURE 3] Product risk score created:")
print(f"  product_risk_score stats:")
print(df['product_risk_score'].describe().round(4).to_string())


[FEATURE 3] Product risk score created:
  product_risk_score stats:
count    590540.0000
mean          0.0350
std           0.0309
min           0.0204
25%           0.0204
50%           0.0204
75%           0.0378
max           0.1169


In [5]:
# ============================================================
# FEATURE 4: CARD-BASED FEATURES
# Source: EDA showed Discover = 7.73%, Credit = 6.68% These are strong signals — engineer risk scores
# ============================================================

# Card network risk score
card4_risk = df.groupby('card4')['isFraud'].mean()
df['card4_risk_score'] = df['card4'].map(card4_risk)

# Card type risk score
card6_risk = df.groupby('card6')['isFraud'].mean()
df['card6_risk_score'] = df['card6'].map(card6_risk)

# Fill any unmapped values
df['card4_risk_score'].fillna(df['isFraud'].mean(), inplace=True)
df['card6_risk_score'].fillna(df['isFraud'].mean(), inplace=True)

print(f"\n[FEATURE 4] Card risk scores created:")
print(f"  card4_risk_score mean: {df['card4_risk_score'].mean():.4f}")
print(f"  card6_risk_score mean: {df['card6_risk_score'].mean():.4f}")


[FEATURE 4] Card risk scores created:
  card4_risk_score mean: 0.0350
  card6_risk_score mean: 0.0350


In [6]:
# ============================================================
# FEATURE 5: TRANSACTION VELOCITY FEATURES
# Business Logic: Fraudsters make multiple transactions in short time windows on same card
# Strategy: Count transactions per card in the dataset as a proxy for velocity
# ============================================================

# Transaction count per card1 (card identifier)
card_txn_count = df.groupby('card1')['TransactionAmt'].transform('count')
df['card1_txn_count'] = card_txn_count

# Mean transaction amount per card — deviation from mean signals fraud
card_amt_mean = df.groupby('card1')['TransactionAmt'].transform('mean')
df['card1_amt_mean']      = card_amt_mean
df['amt_deviation_card1'] = df['TransactionAmt'] - card_amt_mean

# High deviation flag — transaction is 2x above card's average
df['is_high_deviation'] = (
    df['amt_deviation_card1'] > (2 * df.groupby('card1')['TransactionAmt'].transform('std'))
).astype(int)

print(f"\n[FEATURE 5] Velocity features created:")
print(f"  High deviation transactions: {df['is_high_deviation'].sum():,}")
print(f"  card1_txn_count sample:")
print(df['card1_txn_count'].describe().round(2).to_string())


[FEATURE 5] Velocity features created:
  High deviation transactions: 21,867
  card1_txn_count sample:
count    590540.00
mean       2528.82
std        3702.66
min           1.00
25%         132.00
50%         919.00
75%        3152.00
max       14932.00


In [7]:
# ============================================================
# FEATURE 6: C-COLUMN AGGREGATES
# Business Logic: C1-C14 are count-based features (how many addresses, emails linked to card)
# High counts can signal fraudulent accounts
# ============================================================

c_cols = [c for c in df.columns if c.startswith('C') and c != 'card1']
c_cols = [c for c in c_cols if df[c].dtype in [np.float64, np.int64]]

if len(c_cols) > 0:
    df['c_cols_sum']  = df[c_cols].sum(axis=1)
    df['c_cols_mean'] = df[c_cols].mean(axis=1)
    df['c_cols_max']  = df[c_cols].max(axis=1)

    print(f"\n[FEATURE 6] C-column aggregates created from {len(c_cols)} columns:")
    print(f"  c_cols_sum  mean: {df['c_cols_sum'].mean():.2f}")
    print(f"  c_cols_mean mean: {df['c_cols_mean'].mean():.2f}")
    print(f"  c_cols_max  mean: {df['c_cols_max'].mean():.2f}")


[FEATURE 6] C-column aggregates created from 13 columns:
  c_cols_sum  mean: 120.96
  c_cols_mean mean: 9.30
  c_cols_max  mean: 36.89


In [8]:
# ============================================================
# FEATURE 7: V-COLUMN AGGREGATES
# Business Logic: V1-V339 are Vesta-engineered features (transaction behavior signals)
#                 Aggregating reduces 300+ cols to 3 signals
# ============================================================

v_cols = [c for c in df.columns if c.startswith('V')]
v_cols = [c for c in v_cols if df[c].dtype in [np.float64, np.int64]]

if len(v_cols) > 0:
    df['v_cols_sum']  = df[v_cols].sum(axis=1)
    df['v_cols_mean'] = df[v_cols].mean(axis=1)
    df['v_cols_std']  = df[v_cols].std(axis=1)

    print(f"\n[FEATURE 7] V-column aggregates created from {len(v_cols)} columns:")
    print(f"  v_cols_sum  mean: {df['v_cols_sum'].mean():.2f}")
    print(f"  v_cols_mean mean: {df['v_cols_mean'].mean():.2f}")
    print(f"  v_cols_std  mean: {df['v_cols_std'].mean():.2f}")


[FEATURE 7] V-column aggregates created from 157 columns:
  v_cols_sum  mean: 3069.83
  v_cols_mean mean: 19.55
  v_cols_std  mean: 74.85


In [9]:
# ============================================================
# FEATURE 8: INTERACTION FEATURES
# Business Logic: Combining signals creates stronger composite fraud indicators
# ============================================================

# High value + peak hour = highest risk combination
df['high_value_peak_hour'] = (
    df['is_high_value'] & df['is_peak_fraud_hour']
).astype(int)

# High product risk + credit card = compounded risk
df['product_card_risk'] = df['product_risk_score'] * df['card6_risk_score']

# Amount × time of day interaction
df['amt_log_x_hour'] = df['TransactionAmt_log'] * df['hour']

print(f"\n[FEATURE 8] Interaction features created:")
print(f"  high_value_peak_hour transactions: {df['high_value_peak_hour'].sum():,}")
print(f"  product_card_risk mean           : {df['product_card_risk'].mean():.4f}")


[FEATURE 8] Interaction features created:
  high_value_peak_hour transactions: 3,387
  product_card_risk mean           : 0.0013


In [10]:
# ============================================================
# STEP 9: DROP ORIGINAL ENGINEERED SOURCE COLUMNS
# Reason: TransactionDT served its purpose — hour/day extracted. Keeping raw DT adds no value.TransactionID is just an identifier.
# ============================================================

cols_to_drop = ['TransactionDT', 'TransactionID']
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)

print(f"\n[STEP 9] Dropped source columns: {cols_to_drop}")


[STEP 9] Dropped source columns: ['TransactionDT', 'TransactionID']


In [11]:
# ============================================================
# STEP 10: FINAL VALIDATION & SUMMARY
# ============================================================

new_cols     = df.shape[1] - original_cols + len(cols_to_drop)
total_missing = df.isnull().sum().sum()

In [15]:
print(f"Shape Before : (590540,{original_cols})")
print(f"Shape After : {df.shape}")
print(f"New Features Created : {new_cols}")
print(f"Total Missing Values : {total_missing}")

print("="*30)
assert total_missing == 0, f"WARNING: {total_missing} missing values found!"
print("Sanity check passed — zero missing values.")

Shape Before : (590540,204)
Shape After : (590540, 224)
New Features Created : 22
Total Missing Values : 0
Sanity check passed — zero missing values.


In [16]:

# ── Save Engineered Data ──────────────────────────────────
df.to_csv('../data/processed/featured_data.csv', index=False)
print(f"Engineered data saved → data/processed/featured_data.csv")

Engineered data saved → data/processed/featured_data.csv
